In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 0. Reproducibility
# ============================================================

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


# ============================================================
# 1. Load MovieLens 1M
# ============================================================

data_folder = "datasets/movie_lens/"

ratings_file = os.path.join(data_folder, "ratings.dat")
movies_file = os.path.join(data_folder, "movies.dat")
users_file = os.path.join(data_folder, "users.dat")


ratings_df = pd.read_csv(
    ratings_file,
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies_df = pd.read_csv(
    movies_file,
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)

users_df = pd.read_csv(
    users_file,
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"]
)


print("Ratings:", len(ratings_df))
print("Users:", ratings_df["user_id"].nunique())
print("Movies:", ratings_df["movie_id"].nunique())

print(movies_df.head())

Ratings: 1000209
Users: 6040
Movies: 3706
   movie_id                               title                        genres
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy


In [2]:
# ============================================================
# 2. Map raw IDs -> contiguous indices
# ============================================================

unique_users = ratings_df["user_id"].unique()
unique_movies = ratings_df["movie_id"].unique()


user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(unique_users)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(unique_movies)
}


# Reverse mappings
idx_to_user = {
    idx: user_id
    for user_id, idx in user_to_idx.items()
}

idx_to_movie = {
    idx: movie_id
    for movie_id, idx in movie_to_idx.items()
}


ratings_df["user_idx"] = ratings_df["user_id"].map(user_to_idx)
ratings_df["movie_idx"] = ratings_df["movie_id"].map(movie_to_idx)


num_users = len(user_to_idx)
num_movies = len(movie_to_idx)

print("num_users:", num_users)
print("num_movies:", num_movies)

num_users: 6040
num_movies: 3706


In [3]:
# ============================================================
# 3. Chronological split
# ============================================================

ratings_df = ratings_df.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)


ratings_df["rank_from_end"] = (
    ratings_df.groupby("user_id")
    .cumcount(ascending=False)
)


test_df = ratings_df[
    ratings_df["rank_from_end"] == 0
].copy()

val_df = ratings_df[
    ratings_df["rank_from_end"] == 1
].copy()

train_df = ratings_df[
    ratings_df["rank_from_end"] >= 2
].copy()


for df in [train_df, val_df, test_df]:
    df.drop(columns=["rank_from_end"], inplace=True)


print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (988129, 6)
Val: (6040, 6)
Test: (6040, 6)


In [4]:
# ============================================================
# 4. Dataset + DataLoaders
# ============================================================

class MovieLensDataset(Dataset):

    def __init__(self, df):

        self.user_ids = torch.tensor(
            df["user_idx"].values,
            dtype=torch.long
        )

        self.movie_ids = torch.tensor(
            df["movie_idx"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            df["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.user_ids[idx],
            self.movie_ids[idx],
            self.ratings[idx]
        )


train_dataset = MovieLensDataset(train_df)
val_dataset = MovieLensDataset(val_df)
test_dataset = MovieLensDataset(test_df)


train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False
)

In [5]:
# ============================================================
# 5. Simple user/movie embedding + MLP model
# ============================================================

class SimpleMLP(nn.Module):

    def __init__(self, num_users, num_movies, d_embedding):
        super().__init__()

        self.user_emb = nn.Embedding(
            num_users,
            d_embedding
        )

        self.movie_emb = nn.Embedding(
            num_movies,
            d_embedding
        )

        self.mlp = nn.Sequential(
            nn.Linear(d_embedding * 2, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, user_ids, movie_ids):

        # [B, D]
        user_vec = self.user_emb(user_ids)

        # [B, D]
        movie_vec = self.movie_emb(movie_ids)

        # [B, 2D]
        x = torch.cat(
            [user_vec, movie_vec],
            dim=1
        )

        # [B, 1]
        pred = self.mlp(x)

        # [B]
        return pred.squeeze(1)

In [6]:
# ============================================================
# 6. Train recommender
# ============================================================

embedding_dim = 64

model = SimpleMLP(
    num_users=num_users,
    num_movies=num_movies,
    d_embedding=embedding_dim
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for user_batch, movie_batch, rating_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(
            user_batch,
            movie_batch
        )

        loss = criterion(
            predictions,
            rating_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train MSE: {avg_train_loss:.4f}"
    )

Epoch 01 | Train MSE: 0.9944
Epoch 02 | Train MSE: 0.8366
Epoch 03 | Train MSE: 0.8051
Epoch 04 | Train MSE: 0.7755
Epoch 05 | Train MSE: 0.7455
Epoch 06 | Train MSE: 0.7170
Epoch 07 | Train MSE: 0.6901
Epoch 08 | Train MSE: 0.6646
Epoch 09 | Train MSE: 0.6402
Epoch 10 | Train MSE: 0.6165


In [7]:
# ============================================================
# 7. Evaluate recommender
# ============================================================

model.eval()

total_val_loss = 0.0

with torch.no_grad():

    for user_batch, movie_batch, rating_batch in val_loader:

        predictions = model(
            user_batch,
            movie_batch
        )

        loss = criterion(
            predictions,
            rating_batch
        )

        total_val_loss += loss.item()


avg_val_loss = total_val_loss / len(val_loader)

print("Validation MSE:", avg_val_loss)

Validation MSE: 0.922743042310079


In [8]:
# ============================================================
# 8. Extract learned movie embeddings
# ============================================================

movie_vectors = (
    model
    .movie_emb
    .weight
    .detach()
    .clone()
)

print(movie_vectors.shape)

# Expected:
# [num_movies, 64]

torch.Size([3706, 64])


In [9]:
# ============================================================
# 9. Vector Quantizer
# ============================================================

class VectorQuantizer(nn.Module):

    def __init__(self, codebook_size, d_embedding):
        super().__init__()

        # Shape: [K, D]
        self.codebook = nn.Embedding(
            codebook_size,
            d_embedding
        )

    def forward(self, x):

        # x:
        # [B, D]

        # ----------------------------------------------------
        # Compare every vector against every codebook vector
        #
        # x[:, None, :]
        # [B, 1, D]
        #
        # codebook.weight[None, :, :]
        # [1, K, D]
        #
        # diff
        # [B, K, D]
        # ----------------------------------------------------

        diff = (
            x[:, None, :]
            - self.codebook.weight[None, :, :]
        )

        # Squared Euclidean distance
        #
        # [B, K, D]
        #      ↓ sum over D
        # [B, K]

        distances = (diff ** 2).sum(dim=2)

        # For every movie, select closest code
        #
        # [B, K] -> [B]

        code_ids = torch.argmin(
            distances,
            dim=1
        )

        # Retrieve selected vectors
        #
        # [B]

        quantized = self.codebook(code_ids)

        # quantized: [B, D]
        # code_ids:  [B]

        return quantized, code_ids

In [10]:
# ============================================================
# 10. Train Vector Quantizer
# ============================================================

codebook_size = 5

quantizer = VectorQuantizer(
    codebook_size=codebook_size,
    d_embedding=embedding_dim
)


# Fixed input embeddings
quantizer_train = movie_vectors.detach()


# Initialize codebook using actual random movie embeddings
# instead of arbitrary random vectors.

with torch.no_grad():

    random_idx = torch.randperm(
        len(quantizer_train)
    )[:codebook_size]

    quantizer.codebook.weight.copy_(
        quantizer_train[random_idx]
    )


vq_optimizer = torch.optim.Adam(
    quantizer.parameters(),
    lr=0.001
)

vq_criterion = nn.MSELoss()


num_vq_epochs = 100


for epoch in range(num_vq_epochs):

    vq_optimizer.zero_grad()

    quantized, code_ids = quantizer(
        quantizer_train
    )

    # Move codebook vectors toward
    # the movie embeddings assigned to them
    vq_loss = vq_criterion(
        quantized,
        quantizer_train
    )

    vq_loss.backward()
    vq_optimizer.step()


    if epoch % 10 == 0:

        counts = torch.bincount(
            code_ids,
            minlength=codebook_size
        )

        print(
            f"Epoch {epoch:03d} | "
            f"loss={vq_loss.item():.4f} | "
            f"counts={counts.tolist()}"
        )

Epoch 000 | loss=2.8364 | counts=[419, 452, 205, 2065, 565]
Epoch 010 | loss=2.8187 | counts=[420, 451, 204, 2065, 566]
Epoch 020 | loss=2.8012 | counts=[420, 449, 203, 2067, 567]
Epoch 030 | loss=2.7839 | counts=[419, 448, 204, 2066, 569]
Epoch 040 | loss=2.7669 | counts=[419, 446, 204, 2066, 571]
Epoch 050 | loss=2.7502 | counts=[420, 444, 204, 2066, 572]
Epoch 060 | loss=2.7337 | counts=[420, 444, 203, 2066, 573]
Epoch 070 | loss=2.7175 | counts=[416, 444, 201, 2071, 574]
Epoch 080 | loss=2.7015 | counts=[414, 444, 201, 2075, 572]
Epoch 090 | loss=2.6858 | counts=[414, 443, 200, 2074, 575]


In [11]:
# ============================================================
# 11. Assign semantic code to each movie
# ============================================================

quantizer.eval()

with torch.no_grad():

    quantized_movie_vectors, movie_code_ids = quantizer(
        movie_vectors
    )


print(movie_code_ids.shape)

print(
    torch.bincount(
        movie_code_ids,
        minlength=codebook_size
    )
)

torch.Size([3706])
tensor([ 414,  442,  200, 2076,  574])


In [12]:
# ============================================================
# 12. Inspect movies assigned to each code
# ============================================================

for code in range(codebook_size):

    # Internal movie indices assigned to this code
    internal_indices = torch.where(
        movie_code_ids == code
    )[0]


    # Convert internal index -> original MovieLens movie ID
    raw_movie_ids = [
        idx_to_movie[idx.item()]
        for idx in internal_indices
    ]


    # Look up metadata
    code_movies = movies_df[
        movies_df["movie_id"].isin(raw_movie_ids)
    ][
        ["movie_id", "title", "genres"]
    ]


    print()
    print("=" * 80)
    print(
        f"CODE {code} | "
        f"{len(code_movies)} movies"
    )
    print("=" * 80)

    print(
        code_movies.head(20).to_string(index=False)
    )


CODE 0 | 414 movies
 movie_id                                           title                              genres
        4                        Waiting to Exhale (1995)                        Comedy|Drama
       18                               Four Rooms (1995)                            Thriller
       21                               Get Shorty (1995)                 Action|Comedy|Drama
       23                                Assassins (1995)                            Thriller
       34                                     Babe (1995)             Children's|Comedy|Drama
       41                              Richard III (1995)                           Drama|War
       45                               To Die For (1995)                        Comedy|Drama
       80    White Balloon, The (Badkonake Sefid ) (1995)                               Drama
       83 Once Upon a Time... When We Were Colored (1995)                               Drama
      105           Bridges of Madison 

In [13]:
for code in range(codebook_size):

    internal_indices = torch.where(movie_code_ids == code)[0]

    movie_ids = [
        idx_to_movie[idx.item()]
        for idx in internal_indices
    ]

    code_movies = movies_df[
        movies_df["movie_id"].isin(movie_ids)
    ]

    genre_counts = (
        code_movies["genres"]
        .str.split("|")
        .explode()
        .value_counts()
    )

    print(f"\n===== CODE {code} =====")
    print(genre_counts.head(10))


===== CODE 0 =====
genres
Drama          184
Comedy         122
Thriller        64
Romance         48
Action          40
Crime           32
Horror          28
Adventure       22
Sci-Fi          22
Documentary     19
Name: count, dtype: int64

===== CODE 1 =====
genres
Drama          219
Comedy         121
Romance         65
Thriller        54
Action          42
Horror          31
Sci-Fi          30
Adventure       25
Documentary     25
Crime           21
Name: count, dtype: int64

===== CODE 2 =====
genres
Drama          101
Comedy          59
Romance         40
Thriller        22
Action          17
Adventure       15
Sci-Fi          15
Musical         14
Documentary     13
War             12
Name: count, dtype: int64

===== CODE 3 =====
genres
Comedy        718
Drama         695
Action        365
Thriller      297
Horror        248
Romance       230
Adventure     191
Children's    182
Sci-Fi        178
Crime         112
Name: count, dtype: int64

===== CODE 4 =====
genres
Drama      